# Script 3 — Treinamento dos Modelos de ML (V3 Company-Aware)


Objetivo prático:
- manter um modelo global por target,
- mas treinar/validar/avaliar de forma company-aware,
- com métricas e baseline calculadas empresa por empresa,
- mantendo o setor apenas como camada de comparação/diagnóstico.

Correções centrais implementadas:
1) smape_scorer definido corretamente antes do uso.
2) Métricas macro por empresa + pooled + R² within-company.
3) Pesos amostrais por empresa e por target futuro repetido.
4) Seleção de features aprendida apenas no treino, com filtro de colinearidade.
5) Walk-forward reduzido para 3 folds para estabilidade.
6) flag_covid e ano_norm recriados caso não existam.
7) Artefatos mantidos em outputs com nomes compatíveis.

Observação metodológica:
- Eu NÃO vou forçar DFP-only como padrão. O padrão aqui é manter o painel,
  mas reponderar e avaliar por empresa. Se quiser testar DFP-only, basta
  trocar TRAIN_DFP_ONLY = True.


## Etapa 0. Imports e Configuração

In [3]:
import json
import logging
import pickle
import warnings
from pathlib import Path
from datetime import datetime

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 200)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
(PASTA_SAIDA / 'logs').mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_modelagem_company_aware')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'logs' / 'pipeline_modelagem_company_aware.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

SEED = 42
ANO_CORTE = 2022
N_SPLITS_WF = 3
TRAIN_DFP_ONLY = False  # muda para True apenas se quiser comparar a sensibilidade
REMOVE_COLINEAR_FEATURES = True
CORR_DROP_THRESHOLD = 0.8
MAX_FEATURES_PER_TARGET = None  # None = mantém todas após filtro de colinearidade

COVID_ANOS = {2020, 2021}

LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
}
ARCSINH_TARGETS = {
    'TARGET_DFC_MI_6.01',
    'TARGET_DRE_3.11',
}
TARGETS = [
    'TARGET_DRE_3.01',
    'TARGET_DRE_3.11',
    'TARGET_EBITDA',
    'TARGET_BPA_1',
    'TARGET_BPA_1.01',
    'TARGET_BPP_2.01',
    'TARGET_BPP_2.03',
    'TARGET_BPP_2',
    'TARGET_DFC_MI_6.01',
]

logger.info('Script 3 Company-Aware iniciado | sklearn=%s', __import__('sklearn').__version__)
print('✅ Configuração carregada')

2026-05-11 14:55:24 | INFO     | Script 3 Company-Aware iniciado | sklearn=1.8.0


✅ Configuração carregada


## Etapa 1. Carga dos artefatos do Script 2

In [4]:
FEATURES = None
KPIS = None

with open(PASTA_SAIDA / 'features.pkl', 'rb') as f:
    FEATURES = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl', 'rb') as f:
    TARGETS_PRE = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl', 'rb') as f:
    KPIS = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f:
    GRUPOS_TREINO = pickle.load(f)

# Preferência: usar os parquets já gerados pelo Script 2
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste = PASTA_SAIDA / 'teste.parquet'
if not cam_treino.exists() or not cam_teste.exists():
    raise FileNotFoundError(
        'treino.parquet/teste.parquet não encontrados em outputs. '\
        'Execute o Script 2 antes deste Script 3.'
    )

treino = pd.read_parquet(cam_treino)
teste = pd.read_parquet(cam_teste)

# Normalizações mínimas de data
for df in (treino, teste):
    if 'DT_REFER' in df.columns:
        df['DT_REFER'] = pd.to_datetime(df['DT_REFER'], errors='coerce')
    if 'DT_TARGET' in df.columns:
        df['DT_TARGET'] = pd.to_datetime(df['DT_TARGET'], errors='coerce')

logger.info('Split carregado | treino=%s | teste=%s', treino.shape, teste.shape)
print(f'Treino: {treino.shape} | Teste: {teste.shape}')
print(f'ORIGEM treino: {treino["ORIGEM"].value_counts().to_dict() if "ORIGEM" in treino.columns else "N/A"}')
print(f'ORIGEM teste : {teste["ORIGEM"].value_counts().to_dict() if "ORIGEM" in teste.columns else "N/A"}')

# Recria flags e trend features se estiverem ausentes
for df_name, df in [('treino', treino), ('teste', teste)]:
    if 'flag_covid' not in df.columns:
        df['flag_covid'] = df['ANO'].isin(COVID_ANOS).astype(float)
        logger.info('flag_covid recriada em %s', df_name)
    if 'ano_norm' not in df.columns:
        # base temporal simples para capturar tendência estrutural
        df['ano_norm'] = (df['ANO'].astype(float) - 2015.0) / 10.0
        logger.info('ano_norm recriada em %s', df_name)

if 'flag_covid' not in FEATURES:
    FEATURES = list(FEATURES) + ['flag_covid']
if 'ano_norm' not in FEATURES:
    FEATURES = list(FEATURES) + ['ano_norm']

# Anti-leakage prospectivo
anos_treino = set(treino['ANO'].dropna().astype(int).unique()) if 'ANO' in treino.columns else set()
anos_teste = set(teste['ANO'].dropna().astype(int).unique()) if 'ANO' in teste.columns else set()
anos_prosp = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error('Anos prospectivos vazaram para treino/teste: %s', sorted(anos_prosp))
else:
    logger.info('Isolamento prospectivo: PASSOU ✅')

# Diagnóstico de features temporais
colunas_temporais = [f for f in FEATURES if any(s in f for s in ['_lag', '_roll', '_diff1', '_growth1', '_yoy'])]
logger.info('Features temporais: %d/%d', len(colunas_temporais), len(FEATURES))
print(f'Features temporais: {len(colunas_temporais)} de {len(FEATURES)}')

# Filtra o espaço de features só para colunas existentes
FEATURES = [c for c in FEATURES if c in treino.columns]
logger.info('FEATURES finais após interseção com treino: %d', len(FEATURES))
print(f'FEATURES finais: {len(FEATURES)}')

2026-05-11 14:55:28 | INFO     | Split carregado | treino=(713, 681) | teste=(168, 681)
2026-05-11 14:55:28 | INFO     | flag_covid recriada em treino
2026-05-11 14:55:28 | INFO     | ano_norm recriada em treino
2026-05-11 14:55:28 | INFO     | flag_covid recriada em teste
2026-05-11 14:55:28 | INFO     | ano_norm recriada em teste
2026-05-11 14:55:28 | INFO     | Isolamento prospectivo: PASSOU ✅
2026-05-11 14:55:28 | INFO     | Features temporais: 15/17
2026-05-11 14:55:28 | INFO     | FEATURES finais após interseção com treino: 17


Treino: (713, 681) | Teste: (168, 681)
ORIGEM treino: {'ITR': 532, 'DFP': 181}
ORIGEM teste : {'ITR': 144, 'DFP': 24}
Features temporais: 15 de 17
FEATURES finais: 17


## Etapa 2. Métricas, scorer e baseline ingênua

In [5]:
def smape_score(y_true, y_pred):
    """SMAPE em formato de score para GridSearchCV (quanto menor, melhor)."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 1e-9
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else 0.0

# Agora o make_scorer funcionará pois foi importado acima
smape_scorer = make_scorer(smape_score, greater_is_better=False)


def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    mask = denom > 1e-9
    if mask.sum() == 0: return np.nan
    return float(np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]))



def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def r2_seguro(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.isclose(np.var(y_true), 0.0):
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan


def theil_u(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2: return np.nan
    # Erro do modelo vs Erro do Naive (persistência do valor anterior)
    erro_modelo = np.sqrt(np.mean((y_true[1:] - y_pred[1:]) ** 2))
    erro_naive = np.sqrt(np.mean((y_true[1:] - y_true[:-1]) ** 2))
    return float(erro_modelo / erro_naive) if erro_naive > 0 else np.nan

def da_score(y_true, y_pred, y_naive):
    """Directional Accuracy: compara se a direção da mudança foi a mesma."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_naive = np.asarray(y_naive, dtype=float)
    if len(y_true) < 1: return np.nan
    
    mudanca_real = y_true - y_naive
    mudanca_pred = y_pred - y_naive
    # Compara se os sinais das variações são iguais
    return float(np.mean(np.sign(mudanca_real) == np.sign(mudanca_pred)))


def r2_within(y_true, y_pred, companies):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    companies = np.asarray(companies)
    r2s = []
    for co in np.unique(companies):
        m = companies == co
        if m.sum() < 2:
            continue
        ss_res = np.sum((y_true[m] - y_pred[m]) ** 2)
        ss_tot = np.sum((y_true[m] - y_true[m].mean()) ** 2)
        if ss_tot > 0:
            r2s.append(1 - ss_res / ss_tot)
    return float(np.nanmean(r2s)) if r2s else np.nan


def calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='DT_REFER',
                             y_true_col='y_true', y_pred_col='y_pred'):
    # 1. Filtra colunas e remove NaNs
    cols = [c for c in [group_col, time_col, y_true_col, y_pred_col] if c in df_eval.columns]
    df = df_eval[cols].dropna().copy()
    
    if df.empty:
        return {
            'n_obs_validas': 0, 'n_empresas_validas': 0,
            'RMSE_pooled': np.nan, 'MAE_pooled': np.nan, 'SMAPE_pooled': np.nan,
            'R2_pooled': np.nan, 'R2_within': np.nan,
            'RMSE_macro_empresa': np.nan, 'MAE_macro_empresa': np.nan,
            'SMAPE_macro_empresa': np.nan, 'R2_macro_empresa': np.nan,
            'TheilU_macro_empresa': np.nan, 'DA_macro_empresa': np.nan,
        }

    # 2. Ordenação temporal para garantir que o DA e TheilU façam sentido
    if time_col in df.columns:
        df = df.sort_values([group_col, time_col], kind='mergesort')
    else:
        df = df.sort_values([group_col], kind='mergesort')

    yt_all = df[y_true_col].to_numpy(dtype=float)
    yp_all = df[y_pred_col].to_numpy(dtype=float)
    company_all = df[group_col].to_numpy()

    # 3. Cálculo por empresa
    rows = []
    for emp, g in df.groupby(group_col, sort=False):
        yt = g[y_true_col].to_numpy(dtype=float)
        yp = g[y_pred_col].to_numpy(dtype=float)
        
        # DEFINIÇÃO DE N (Para evitar o NameError)
        n = len(yt)
        
        rows.append({
            'empresa': emp,
            'n_obs': n,
            'RMSE': rmse(yt, yp),
            'MAE': float(mean_absolute_error(yt, yp)),
            'SMAPE': smape(yt, yp),
            'R2': r2_seguro(yt, yp),
            'TheilU': theil_u(yt, yp),
            # O alinhamento correto para o DA: 
            # yt[1:] é o valor atual, yt[:-1] é o valor anterior (naive)
            'DA': da_score(yt[1:], yp[1:], yt[:-1]) if n > 1 else np.nan,
        })
    
    per_emp = pd.DataFrame(rows)
    
    # Dentro de calcular_metricas_painel, antes do return:
    per_emp['RMSE'] = per_emp['RMSE'].clip(upper=per_emp['RMSE'].quantile(0.95))
    per_emp['SMAPE'] = per_emp['SMAPE'].clip(upper=0.99) # SMAPE não deve passar de 100%

    # 4. Agregação final
    return {
        'n_obs_validas': int(len(df)),
        'n_empresas_validas': int(len(per_emp)),
        'RMSE_pooled': rmse(yt_all, yp_all),
        'MAE_pooled': float(mean_absolute_error(yt_all, yp_all)),
        'SMAPE_pooled': smape(yt_all, yp_all),
        'R2_pooled': r2_seguro(yt_all, yp_all),
        'R2_within': r2_within(yt_all, yp_all, company_all),
        'RMSE_macro_empresa': float(per_emp['RMSE'].mean()) if not per_emp.empty else np.nan,
        'MAE_macro_empresa': float(per_emp['MAE'].mean()) if not per_emp.empty else np.nan,
        'SMAPE_macro_empresa': float(per_emp['SMAPE'].mean()) if not per_emp.empty else np.nan,
        'R2_macro_empresa': float(per_emp['R2'].mean()) if not per_emp.empty else np.nan,
        'TheilU_macro_empresa': float(per_emp['TheilU'].mean()) if not per_emp.empty else np.nan,
        'DA_macro_empresa': float(per_emp['DA'].mean()) if not per_emp.empty else np.nan,
    }


def calcular_baseline(treino_df, teste_df, target):
    """Persistência do último valor observado da própria empresa."""
    if target not in treino_df.columns or target not in teste_df.columns:
        return {}
    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns:
        return {}

    candidatos_tempo = [
        'DT_REFER', 'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next((c for c in candidatos_tempo if c in treino_df.columns and c in teste_df.columns), None)

    cols_ord = ['CNPJ_CIA']
    if time_col is not None:
        cols_ord.append(time_col)

    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original'] = np.arange(len(teste_tmp))

    cols_select = list(dict.fromkeys(cols_ord + ['_ordem_original', target]))
    base = pd.concat([
        treino_tmp[cols_select].assign(__split='treino'),
        teste_tmp[cols_select].assign(__split='teste'),
    ], ignore_index=True)

    base = base.sort_values(cols_ord + ['_ordem_original'], kind='mergesort').reset_index(drop=True)
    base['baseline_prev'] = base.groupby('CNPJ_CIA')[target].transform(lambda s: s.ffill().shift(1))

    mask_teste = base['__split'] == 'teste'
    mask_valido = mask_teste & base[target].notna() & base['baseline_prev'].notna()
    if mask_valido.sum() == 0:
        return {}

    df_eval = base.loc[mask_valido, ['CNPJ_CIA', '_ordem_original', target, 'baseline_prev']].copy()
    df_eval = df_eval.rename(columns={target: 'y_true', 'baseline_prev': 'y_pred'})
    m = calcular_metricas_painel(df_eval, group_col='CNPJ_CIA', time_col='_ordem_original')
    m['Cobertura_baseline'] = float(mask_valido.sum() / max(1, int(mask_teste.sum())))
    m['TimeCol_baseline'] = time_col if time_col is not None else ''
    return m


baselines = {}
print('=== Baseline Ingênua por empresa (persistência) ===')
print(f"  {'Target':<30} {'RMSEm':>14} {'SMAPEm':>8} {'DAm':>6} {'U':>7} {'Cob.':>6}")
print(f"  {'-'*30} {'-'*14} {'-'*8} {'-'*6} {'-'*7} {'-'*6} {'-'*14}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_macro_empresa']:>14,.0f} "
              f"{b['SMAPE_macro_empresa']:>8.1%} {b['DA_macro_empresa']:>6.1%} "
              f"{b['TheilU_macro_empresa']:>7.2f} {b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} {'N/A':>14} {'N/A':>8} {'N/A':>6} {'N/A':>7} {'N/A':>6}  N/A")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                  RMSEm   SMAPEm    DAm       U   Cob.
  ------------------------------ -------------- -------- ------ ------- ------ --------------
  TARGET_DRE_3.01                     1,431,829     1.5%  84.8%    1.00 100.0%   DT_REFER
  TARGET_DRE_3.11                     1,097,063     9.4%  84.8%    1.00 100.0%   DT_REFER
  TARGET_EBITDA                         803,890     2.0%  88.1%    1.00 100.0%   DT_REFER
  TARGET_BPA_1                        3,512,994     2.1%  84.8%    1.00 100.0%   DT_REFER
  TARGET_BPA_1.01                     1,028,705     2.4%  84.8%    1.00 100.0%   DT_REFER
  TARGET_BPP_2.01                       863,671     2.6%  84.8%    1.00 100.0%   DT_REFER
  TARGET_BPP_2.03                     1,605,865     2.7%  84.8%    1.00 100.0%   DT_REFER
  TARGET_BPP_2                        3,512,994     2.1%  84.8%    1.00 100.0%   DT_REFER
  TARGET_DFC_MI_6.01                  1,061,590     5.7

## Etapa 3. Caminho temporal, pesos por empresa e seleção de features

In [6]:
def criar_folds_walkforward(df, time_col='ANO', n_splits=N_SPLITS_WF, min_train_periods=2):
    if time_col not in df.columns:
        time_col = 'DT_REFER' if 'DT_REFER' in df.columns else None
    if time_col is None:
        logger.warning('Walk-Forward: nenhuma coluna temporal disponível.')
        return []

    serie_tempo = df[time_col]
    periodos = pd.Index(pd.unique(serie_tempo.dropna())).sort_values()
    if len(periodos) <= min_train_periods:
        logger.warning('Walk-Forward: períodos insuficientes para criar folds.')
        return []

    max_folds = len(periodos) - min_train_periods
    if n_splits > max_folds:
        n_splits = max(1, max_folds)
        logger.warning('Walk-Forward: reduzindo para %d folds', n_splits)

    periodos_validacao = periodos[-n_splits:]
    folds = []
    for p_val in periodos_validacao:
        idx_tr = np.where(serie_tempo.values < p_val)[0]
        idx_val = np.where(serie_tempo.values == p_val)[0]
        if len(idx_tr) > 0 and len(idx_val) > 0:
            folds.append((idx_tr, idx_val))
    logger.info('Walk-Forward CV: %d folds | validação: %s', len(folds), [str(p) for p in periodos_validacao])
    return folds


def calcular_pesos_amostra(df, group_col='CNPJ_CIA', future_col='DT_TARGET'):
    """
    Peso inverso por empresa e por futuro repetido.
    - Equaliza empresas.
    - Evita que o mesmo target futuro repetido nas linhas ITR domine o treino.
    """
    n = len(df)
    if n == 0:
        return np.array([], dtype=float)

    if group_col not in df.columns:
        return np.ones(n, dtype=float)

    cont_emp = df[group_col].value_counts()
    w_emp = 1.0 / df[group_col].map(cont_emp).astype(float)

    if future_col in df.columns:
        key = df[group_col].astype(str) + '|' + df[future_col].astype(str)
        cont_fut = key.value_counts()
        w_fut = 1.0 / key.map(cont_fut).astype(float)
    else:
        w_fut = 1.0

    pesos = np.asarray(w_emp * w_fut, dtype=float)
    pesos = pesos / np.nanmean(pesos)
    return pesos


def get_target_transform(target):
    if target in LOG_TARGETS:
        return 'log1p'
    if target in ARCSINH_TARGETS:
        return 'arcsinh'
    return 'none'


def target_transform(y, transformacao='none'):
    y_arr = np.asarray(y, dtype=float)
    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(f'target_transform: há {n_bad} valores não finitos.')
    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            raise ValueError("target_transform(log1p): valores <= -1 encontrados. Use 'arcsinh'.")
        return np.log1p(y_arr)
    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)
    return y_arr.copy()


def target_inverse_transform(y_pred, transformacao='none'):
    y_arr = np.asarray(y_pred, dtype=float)
    if transformacao == 'log1p':
        return np.expm1(y_arr)
    if transformacao == 'arcsinh':
        return np.sinh(y_arr)
    return y_arr


def selecionar_features_colineares(df_train, candidate_features, target_col, threshold=CORR_DROP_THRESHOLD):
    """
    Seleção simples, treino-only, para reduzir multicolinearidade.
    Ordena por |corr(feature,target)| e remove pares com correlação acima do limiar.
    """
    cols = [c for c in candidate_features if c in df_train.columns]
    tmp = df_train[cols + [target_col]].dropna().copy()
    if tmp.empty or len(cols) == 0:
        return cols

    corr_target = tmp[cols].corrwith(tmp[target_col]).abs().fillna(0.0).sort_values(ascending=False)
    ordered = corr_target.index.tolist()
    corr_mat = tmp[cols].corr().abs().fillna(0.0)

    kept = []
    for feat in ordered:
        if feat not in corr_mat.columns:
            continue
        if all(corr_mat.loc[feat, k] <= threshold for k in kept):
            kept.append(feat)
        if MAX_FEATURES_PER_TARGET is not None and len(kept) >= MAX_FEATURES_PER_TARGET:
            break

    if len(kept) == 0:
        kept = ordered[: min(20, len(ordered))]
    return kept


# Algoritmos
est_ridge = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('ridge', Ridge()),
])
grade_ridge = {'ridge__alpha': [10.0, 100.0, 1000.0]}

est_svr = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('svr', SVR(kernel='rbf', max_iter=20000)),
])
grade_svr = {
    'svr__C': [0.1, 1.0, 10.0],
    'svr__epsilon': [0.05, 0.1, 0.5],
    'svr__gamma': ['scale', 'auto'],
}

est_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('rf', RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)),
])
grade_rf = {
    'rf__max_depth': [None, 10, 20],
    'rf__min_samples_leaf': [1, 2, 5],
    'rf__max_features': ['sqrt', 'log2'],
}

est_gb = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('gb', GradientBoostingRegressor(random_state=SEED)),
])
grade_gb = {
    'gb__n_estimators': [100, 200, 300],
    'gb__learning_rate': [0.03, 0.05, 0.1],
    'gb__max_depth': [3, 5, 7],
    'gb__subsample': [0.8, 1.0],
}

ALGORITMOS = {
    'Ridge': (est_ridge, grade_ridge),
    'SVR': (est_svr, grade_svr),
    'RandomForest': (est_rf, grade_rf),
    'GradientBoosting': (est_gb, grade_gb),
}

logger.info('%d algoritmos configurados | Walk-Forward n_splits=%d', len(ALGORITMOS), N_SPLITS_WF)
print(f'✅ {len(ALGORITMOS)} algoritmos configurados')

2026-05-11 14:55:38 | INFO     | 4 algoritmos configurados | Walk-Forward n_splits=3


✅ 4 algoritmos configurados


## Etapa 4. Treinamento com Walk-Forward nested CV

In [7]:
def treinar_alg(nome, estimador, grade, df_treino_completo, target, features,
                transformacao='none', n_splits_wf=N_SPLITS_WF,
                group_col='CNPJ_CIA', time_col='ANO'):
    """
    Treinamento company-aware:
    - pesos por empresa e por futuro repetido;
    - walk-forward temporal por ano;
    - scoring por SMAPE;
    - métricas macro por empresa.
    """
    use_cols = [c for c in features + [group_col, target] if c in df_treino_completo.columns]
    if time_col in df_treino_completo.columns:
        use_cols += [time_col]
    if 'DT_REFER' in df_treino_completo.columns:
        use_cols += ['DT_REFER']
    if 'DT_TARGET' in df_treino_completo.columns:
        use_cols += ['DT_TARGET']

    use_cols = list(dict.fromkeys(use_cols))
    df_t = df_treino_completo[use_cols].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)

    if TRAIN_DFP_ONLY and 'ORIGEM' in df_t.columns:
        df_t = df_t[df_t['ORIGEM'] == 'DFP'].copy().reset_index(drop=True)

    if time_col not in df_t.columns:
        time_col = 'DT_REFER' if 'DT_REFER' in df_t.columns else ('ANO' if 'ANO' in df_t.columns else None)

    X_full = df_t[features].values
    y_full = df_t[target].values
    y_fit_full = target_transform(y_full, transformacao)

    sample_weight_full = calcular_pesos_amostra(df_t, group_col=group_col, future_col='DT_TARGET')
    final_step = list(estimador.named_steps.keys())[-1]
    fit_params_full = {f'{final_step}__sample_weight': sample_weight_full}

    folds_ext = criar_folds_walkforward(df_t, time_col=time_col or 'ANO', n_splits=n_splits_wf)
    if len(folds_ext) < 2:
        logger.warning('%s | %s: folds insuficientes, fallback cv=3', nome, target)
        gs_fb = GridSearchCV(estimador, grade, cv=3, scoring=smape_scorer,
                             refit=True, n_jobs=-1, verbose=0)
        gs_fb.fit(X_full, y_fit_full, **fit_params_full)
        best_est = gs_fb.best_estimator_
        metricas = {
            'RMSE_CV_macro_empresa': np.nan,
            'RMSE_CV_macro_empresa_std': np.nan,
            'MAE_CV_macro_empresa': np.nan,
            'SMAPE_CV_macro_empresa': np.nan,
            'SMAPE_CV_macro_empresa_std': np.nan,
            'R2_CV_macro_empresa': np.nan,
            'R2_CV_pooled': np.nan,
            'R2_within_CV': np.nan,
            'TheilU_CV_macro_empresa': np.nan,
            'DA_CV_macro_empresa': np.nan,
            'RMSE_CV_pooled': np.nan,
            'SMAPE_CV_pooled': np.nan,
            'transformacao': transformacao,
            'log_transform': transformacao == 'log1p',
            'best_params': gs_fb.best_params_,
            'n_folds_wf': 0,
            'selected_features': features,
        }
        return best_est, metricas

    rmse_macro_v, mae_macro_v, smape_macro_v, r2_macro_v, theil_macro_v, da_macro_v = [], [], [], [], [], []
    rmse_pool_v, smape_pool_v, r2_pool_v, r2_within_v = [], [], [], []

    for tr_idx_ext, val_idx_ext in folds_ext:
        X_tr_ext = X_full[tr_idx_ext]
        y_tr_ext = y_fit_full[tr_idx_ext]
        y_val_orig = y_full[val_idx_ext]

        df_sub = df_t.iloc[tr_idx_ext].reset_index(drop=True)
        folds_int = criar_folds_walkforward(df_sub, time_col=time_col or 'ANO', n_splits=max(2, n_splits_wf - 1))
        cv_int = folds_int if len(folds_int) >= 2 else 3

        w_tr_ext = sample_weight_full[tr_idx_ext]
        fit_params_tr = {f'{final_step}__sample_weight': w_tr_ext}

        gs = GridSearchCV(estimador, grade, cv=cv_int, scoring=smape_scorer,
                          refit=True, n_jobs=-1, verbose=0)
        gs.fit(X_tr_ext, y_tr_ext, **fit_params_tr)
        melhor_fold = gs.best_estimator_

        y_pred_raw = melhor_fold.predict(X_full[val_idx_ext])
        y_pred = target_inverse_transform(y_pred_raw, transformacao)

        df_fold_eval = df_t.iloc[val_idx_ext][[group_col]].copy()
        if time_col in df_t.columns:
            df_fold_eval[time_col] = df_t.iloc[val_idx_ext][time_col].values
        df_fold_eval['y_true'] = y_val_orig
        df_fold_eval['y_pred'] = y_pred

        m_fold = calcular_metricas_painel(df_fold_eval, group_col=group_col,
                                          time_col=time_col or group_col,
                                          y_true_col='y_true', y_pred_col='y_pred')
        rmse_macro_v.append(m_fold['RMSE_macro_empresa'])
        mae_macro_v.append(m_fold['MAE_macro_empresa'])
        smape_macro_v.append(m_fold['SMAPE_macro_empresa'])
        r2_macro_v.append(m_fold['R2_macro_empresa'])
        theil_macro_v.append(m_fold['TheilU_macro_empresa'])
        da_macro_v.append(m_fold['DA_macro_empresa'])
        rmse_pool_v.append(m_fold['RMSE_pooled'])
        smape_pool_v.append(m_fold['SMAPE_pooled'])
        r2_pool_v.append(m_fold['R2_pooled'])
        r2_within_v.append(m_fold['R2_within'])

    gs_final = GridSearchCV(estimador, grade, cv=folds_ext if len(folds_ext) >= 2 else 3,
                            scoring=smape_scorer, refit=True, n_jobs=-1, verbose=0)
    gs_final.fit(X_full, y_fit_full, **fit_params_full)
    best_est = gs_final.best_estimator_

    def _m(lst):
        return float(np.nanmean(lst))
    def _s(lst):
        return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV_macro_empresa': _m(rmse_macro_v),
        'RMSE_CV_macro_empresa_std': _s(rmse_macro_v),
        'MAE_CV_macro_empresa': _m(mae_macro_v),
        'SMAPE_CV_macro_empresa': _m(smape_macro_v),
        'SMAPE_CV_macro_empresa_std': _s(smape_macro_v),
        'R2_CV_macro_empresa': _m(r2_macro_v),
        'R2_CV_pooled': _m(r2_pool_v),
        'R2_within_CV': _m(r2_within_v),
        'TheilU_CV_macro_empresa': _m(theil_macro_v),
        'DA_CV_macro_empresa': _m(da_macro_v),
        'RMSE_CV_pooled': _m(rmse_pool_v),
        'SMAPE_CV_pooled': _m(smape_pool_v),
        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params': gs_final.best_params_,
        'n_folds_wf': len(folds_ext),
        'selected_features': features,
    }

    flag_theil = '✅' if metricas['TheilU_CV_macro_empresa'] < 1 else '⚠️'
    logger.info(
        '  %-20s RMSEm=%10.0f±%8.0f  SMAPEm=%5.1f%%  R2m=%5.3f  U=%s%.3f  DAm=%.1f%%  folds=%d',
        nome,
        metricas['RMSE_CV_macro_empresa'], metricas['RMSE_CV_macro_empresa_std'],
        metricas['SMAPE_CV_macro_empresa'] * 100, metricas['R2_CV_macro_empresa'],
        flag_theil, metricas['TheilU_CV_macro_empresa'],
        metricas['DA_CV_macro_empresa'] * 100, metricas['n_folds_wf']
    )
    print(
        f"  {flag_theil} {nome:<20} RMSEm={metricas['RMSE_CV_macro_empresa']:>12,.0f}  "
        f"SMAPEm={metricas['SMAPE_CV_macro_empresa']:>5.1%}  R²m={metricas['R2_CV_macro_empresa']:>6.3f}  "
        f"U={metricas['TheilU_CV_macro_empresa']:.3f}  DAm={metricas['DA_CV_macro_empresa']:.1%}  folds={metricas['n_folds_wf']}"
    )
    return best_est, metricas


## Etapa 5. Loop principal por target

In [8]:
resultados = {}
metricas_teste = {}
feature_importances = {}
modelos_finais = {}
selected_features_por_target = {}

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\n{'='*80}")
    print(f"TARGET: {target} | transform={transformacao}")
    if b:
        print(f"Baseline → RMSEm={b.get('RMSE_macro_empresa', np.nan):,.0f}  SMAPEm={b.get('SMAPE_macro_empresa', np.nan):.1%}  "
              f"R²m={b.get('R2_macro_empresa', np.nan):.3f}  DAm={b.get('DA_macro_empresa', np.nan):.1%}  Cob={b.get('Cobertura_baseline', np.nan):.1%}")

    # Seleção treino-only de features para este target
    base_features = [c for c in FEATURES if c in treino.columns and c != target]
    train_for_sel = treino[base_features + [target]].copy()
    selected_features = selecionar_features_colineares(train_for_sel, base_features, target, threshold=CORR_DROP_THRESHOLD)
    selected_features_por_target[target] = selected_features

    print(f"Features selecionadas: {len(selected_features)}")

    resultados[target] = {}
    metricas_teste[target] = {}

    for nome, (est, grade) in ALGORITMOS.items():
        modelo, met_cv = treinar_alg(
            nome=nome,
            estimador=est,
            grade=grade,
            df_treino_completo=treino,
            target=target,
            features=selected_features,
            transformacao=transformacao,
            n_splits_wf=N_SPLITS_WF,
            group_col='CNPJ_CIA',
            time_col='ANO',
        )
        resultados[target][nome] = (modelo, met_cv)
        modelos_finais[(target, nome)] = modelo

        joblib.dump(
            {
                'modelo': modelo,
                'transformacao': transformacao,
                'log_transform': transformacao == 'log1p',
                'features': selected_features,
                'target': target,
                'selected_features': selected_features,
            },
            PASTA_SAIDA / f'modelo_{target}_{nome}.pkl'
        )

    logger.info('TARGET %s concluído', target)

print('\n✅ Treinamento concluído para todos os targets.')


2026-05-11 14:55:49 | INFO     | Walk-Forward CV: 3 folds | validação: ['2020', '2021', '2022']
2026-05-11 14:55:49 | INFO     | Walk-Forward CV: 2 folds | validação: ['2018', '2019']



TARGET: TARGET_DRE_3.01 | transform=log1p
Baseline → RMSEm=1,431,829  SMAPEm=1.5%  R²m=0.409  DAm=84.8%  Cob=100.0%
Features selecionadas: 5


2026-05-11 14:55:54 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']
2026-05-11 14:55:56 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-11 14:55:56 | INFO     |   Ridge                RMSEm=  48144446±15879118  SMAPEm= 71.0%  R2m=-10443.223  U=⚠️31.873  DAm=15.6%  folds=3
2026-05-11 14:55:56 | INFO     | Walk-Forward CV: 3 folds | validação: ['2020', '2021', '2022']
2026-05-11 14:55:56 | INFO     | Walk-Forward CV: 2 folds | validação: ['2018', '2019']
2026-05-11 14:55:57 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']


  ⚠️ Ridge                RMSEm=  48,144,446  SMAPEm=71.0%  R²m=-10443.223  U=31.873  DAm=15.6%  folds=3


2026-05-11 14:55:57 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-11 14:55:57 | INFO     |   SVR                  RMSEm=  35424098± 5898202  SMAPEm= 50.8%  R2m=-1697.672  U=⚠️17.827  DAm=16.1%  folds=3
2026-05-11 14:55:57 | INFO     | Walk-Forward CV: 3 folds | validação: ['2020', '2021', '2022']
2026-05-11 14:55:57 | INFO     | Walk-Forward CV: 2 folds | validação: ['2018', '2019']


...

  ⚠️ SVR                  RMSEm=   8,288,315  SMAPEm=55.9%  R²m=-23683.278  U=25.488  DAm=20.7%  folds=3


2026-05-11 15:01:43 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']
2026-05-11 15:01:48 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-11 15:02:01 | INFO     |   RandomForest         RMSEm=   4347542± 1684153  SMAPEm= 53.4%  R2m=-5425.169  U=⚠️11.354  DAm=17.9%  folds=3
2026-05-11 15:02:01 | INFO     | Walk-Forward CV: 3 folds | validação: ['2020', '2021', '2022']
2026-05-11 15:02:01 | INFO     | Walk-Forward CV: 2 folds | validação: ['2018', '2019']


  ⚠️ RandomForest         RMSEm=   4,347,542  SMAPEm=53.4%  R²m=-5425.169  U=11.354  DAm=17.9%  folds=3


2026-05-11 15:02:05 | INFO     | Walk-Forward CV: 2 folds | validação: ['2019', '2020']
2026-05-11 15:02:09 | INFO     | Walk-Forward CV: 2 folds | validação: ['2020', '2021']
2026-05-11 15:02:20 | INFO     |   GradientBoosting     RMSEm=   3468966± 1096074  SMAPEm= 45.5%  R2m=-993.269  U=⚠️7.400  DAm=18.4%  folds=3
2026-05-11 15:02:20 | INFO     | TARGET TARGET_DFC_MI_6.01 concluído


  ⚠️ GradientBoosting     RMSEm=   3,468,966  SMAPEm=45.5%  R²m=-993.269  U=7.400  DAm=18.4%  folds=3

✅ Treinamento concluído para todos os targets.


## Etapa 6. Avaliação no teste hold-out

In [9]:
def avaliar_teste(modelo, df_eval, features, target, transformacao, group_col='CNPJ_CIA', time_col='DT_REFER'):
    cols = [c for c in features + [target, group_col] if c in df_eval.columns]
    if time_col in df_eval.columns:
        cols += [time_col]
    cols = list(dict.fromkeys(cols))
    df = df_eval[cols].copy()
    df = df[df[target].notna()].reset_index(drop=True)

    y_pred_raw = modelo.predict(df[features].values)
    y_pred = target_inverse_transform(y_pred_raw, transformacao)

    df_out = df[[group_col]].copy()
    if time_col in df.columns:
        df_out[time_col] = df[time_col].values
    df_out['y_true'] = df[target].values
    df_out['y_pred'] = y_pred

    return calcular_metricas_painel(df_out, group_col=group_col,
                                    time_col=time_col if time_col in df_out.columns else group_col,
                                    y_true_col='y_true', y_pred_col='y_pred')


predicoes_teste_detalhadas = []
print('\n=== Avaliação no Teste Hold-out (2023–2024) ===')
for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})
    selected_features = selected_features_por_target[target]

    df_te = teste[selected_features + [target, 'CNPJ_CIA'] + (["DT_REFER"] if 'DT_REFER' in teste.columns else [])].copy()
    df_te = df_te[df_te[target].notna()].copy()

    baseline_rmse = b.get('RMSE_macro_empresa', np.inf)
    print(f"\n{target} (baseline RMSEm={baseline_rmse:,.0f}  DAm={b.get('DA_macro_empresa', 0):.1%}  Cob={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSEm':>14} {'SMAPEm':>8} {'R²m':>7} {'U':>7} {'DAm':>7} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*8} {'-'*7} {'-'*7} {'-'*7} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, df_te, selected_features, target, transformacao)
        metricas_teste[target][nome] = m
        bateu = m['RMSE_macro_empresa'] < baseline_rmse
        theil_ok = (m['TheilU_macro_empresa'] or 1.0) < 1.0
        flag = '✅' if bateu and theil_ok else ('🟡' if bateu else '❌')

        print(
            f"  {flag} {nome:<18} {m['RMSE_macro_empresa']:>14,.0f} {m['SMAPE_macro_empresa']:>8.1%} "
            f"{m['R2_macro_empresa']:>7.3f} {m['TheilU_macro_empresa']:>7.3f} {m['DA_macro_empresa']:>7.1%} {'✅' if bateu else '❌':>7}"
        )
        logger.info('Teste | %s | %s: RMSEm=%.0f SMAPEm=%.2f%% R2m=%.3f TheilU=%.3f DAm=%.1f%%',
                    target, nome,
                    m['RMSE_macro_empresa'], m['SMAPE_macro_empresa'] * 100,
                    m['R2_macro_empresa'], m['TheilU_macro_empresa'], m['DA_macro_empresa'] * 100)

        # Guarda previsão detalhada por linha para inspeção posterior
        y_pred_raw = modelo.predict(df_te[selected_features].values)
        y_pred = target_inverse_transform(y_pred_raw, transformacao)
        aux = df_te[['CNPJ_CIA'] + (["DT_REFER"] if 'DT_REFER' in df_te.columns else [])].copy()
        aux['Target'] = target
        aux['Algoritmo'] = nome
        aux['y_true'] = df_te[target].values
        aux['y_pred'] = y_pred
        aux['erro'] = aux['y_true'] - aux['y_pred']
        predicoes_teste_detalhadas.append(aux)

        # Feature importance do melhor modelo será definido depois; este bloco só calcula tudo

2026-05-11 15:05:57 | INFO     | Teste | TARGET_DRE_3.01 | Ridge: RMSEm=39142194 SMAPEm=67.16% R2m=-45971.511 TheilU=80.931 DAm=6.6%
2026-05-11 15:05:57 | INFO     | Teste | TARGET_DRE_3.01 | SVR: RMSEm=32808834 SMAPEm=39.24% R2m=-1774.516 TheilU=29.656 DAm=9.3%
2026-05-11 15:05:57 | INFO     | Teste | TARGET_DRE_3.01 | RandomForest: RMSEm=9514684 SMAPEm=9.85% R2m=-33.123 TheilU=4.803 DAm=9.3%



=== Avaliação no Teste Hold-out (2023–2024) ===

TARGET_DRE_3.01 (baseline RMSEm=1,431,829  DAm=84.8%  Cob=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                  39,142,194    67.2% -45971.511  80.931    6.6%       ❌
  ❌ SVR                    32,808,834    39.2% -1774.516  29.656    9.3%       ❌
  ❌ RandomForest            9,514,684     9.8% -33.123   4.803    9.3%       ❌


2026-05-11 15:05:58 | INFO     | Teste | TARGET_DRE_3.01 | GradientBoosting: RMSEm=2548621 SMAPEm=4.35% R2m=-13.373 TheilU=2.465 DAm=13.9%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_DRE_3.11 | Ridge: RMSEm=4346133 SMAPEm=86.93% R2m=-7840.571 TheilU=31.862 DAm=6.0%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_DRE_3.11 | SVR: RMSEm=3500242 SMAPEm=45.48% R2m=-612.435 TheilU=13.383 DAm=6.0%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_DRE_3.11 | RandomForest: RMSEm=2169951 SMAPEm=37.67% R2m=-294.681 TheilU=8.076 DAm=9.9%


  ❌ GradientBoosting        2,548,621     4.3% -13.373   2.465   13.9%       ❌

TARGET_DRE_3.11 (baseline RMSEm=1,097,063  DAm=84.8%  Cob=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                   4,346,133    86.9% -7840.571  31.862    6.0%       ❌
  ❌ SVR                     3,500,242    45.5% -612.435  13.383    6.0%       ❌
  ❌ RandomForest            2,169,951    37.7% -294.681   8.076    9.9%       ❌


2026-05-11 15:05:58 | INFO     | Teste | TARGET_DRE_3.11 | GradientBoosting: RMSEm=1671510 SMAPEm=18.46% R2m=-590.755 TheilU=8.955 DAm=9.9%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_EBITDA | Ridge: RMSEm=12305505 SMAPEm=73.70% R2m=-901.707 TheilU=23.383 DAm=4.6%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_EBITDA | SVR: RMSEm=9719003 SMAPEm=40.06% R2m=-361.405 TheilU=16.239 DAm=5.2%
2026-05-11 15:05:58 | INFO     | Teste | TARGET_EBITDA | RandomForest: RMSEm=5728943 SMAPEm=29.00% R2m=-192.048 TheilU=9.982 DAm=4.6%


...

  ❌ GradientBoosting        3,481,471    11.9% -26.994   3.369   10.0%       ❌

TARGET_BPP_2 (baseline RMSEm=3,512,994  DAm=84.8%  Cob=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                  41,969,723    70.2% -8296.508  57.741    4.7%       ❌
  ❌ SVR                    45,437,438    31.2% -1947.727  30.980    4.0%       ❌
  ❌ RandomForest           14,587,455    14.6% -130.906   7.179    8.7%       ❌


2026-05-11 15:05:59 | INFO     | Teste | TARGET_BPP_2 | GradientBoosting: RMSEm=5665592 SMAPEm=4.52% R2m=-9.333 TheilU=2.573 DAm=9.3%
2026-05-11 15:05:59 | INFO     | Teste | TARGET_DFC_MI_6.01 | Ridge: RMSEm=8561409 SMAPEm=97.54% R2m=-945913.926 TheilU=287.573 DAm=7.9%
2026-05-11 15:06:00 | INFO     | Teste | TARGET_DFC_MI_6.01 | SVR: RMSEm=6067325 SMAPEm=41.13% R2m=-1224.125 TheilU=20.524 DAm=7.2%
2026-05-11 15:06:00 | INFO     | Teste | TARGET_DFC_MI_6.01 | RandomForest: RMSEm=2577496 SMAPEm=38.81% R2m=-171.221 TheilU=9.402 DAm=7.9%


  ❌ GradientBoosting        5,665,592     4.5%  -9.333   2.573    9.3%       ❌

TARGET_DFC_MI_6.01 (baseline RMSEm=1,061,590  DAm=84.8%  Cob=100.0%)
  Algoritmo                     RMSEm   SMAPEm     R²m       U     DAm  Bateu?
  -------------------- -------------- -------- ------- ------- ------- -------
  ❌ Ridge                   8,561,409    97.5% -945913.926 287.573    7.9%       ❌
  ❌ SVR                     6,067,325    41.1% -1224.125  20.524    7.2%       ❌
  ❌ RandomForest            2,577,496    38.8% -171.221   9.402    7.9%       ❌


2026-05-11 15:06:00 | INFO     | Teste | TARGET_DFC_MI_6.01 | GradientBoosting: RMSEm=1658695 SMAPEm=26.63% R2m=-106.599 TheilU=8.006 DAm=8.6%


  ❌ GradientBoosting        1,658,695    26.6% -106.599   8.006    8.6%       ❌


## Etapa 7. Seleção do melhor modelo por target

In [10]:
def escolher_melhor_modelo_cv(resultados_target):
    return min(
        resultados_target.items(),
        key=lambda item: (
            item[1][1].get('SMAPE_CV_macro_empresa', np.inf),
            item[1][1].get('TheilU_CV_macro_empresa', np.inf),
            item[1][1].get('RMSE_CV_macro_empresa', np.inf),
        )
    )[0]


melhores = {t: escolher_melhor_modelo_cv(resultados[t]) for t in TARGETS}
print('\n=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===')
for t, alg in melhores.items():
    m_cv = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    print(f"  {t:<35} → {alg:<18} SMAPE_CV={m_cv['SMAPE_CV_macro_empresa']:.1%} | SMAPE_teste={m_test['SMAPE_macro_empresa']:.1%} | U_teste={m_test['TheilU_macro_empresa']:.3f}")



=== Melhor modelo por target (critério: SMAPE_CV macro por empresa) ===
  TARGET_DRE_3.01                     → GradientBoosting   SMAPE_CV=8.1% | SMAPE_teste=4.3% | U_teste=2.465
  TARGET_DRE_3.11                     → GradientBoosting   SMAPE_CV=33.3% | SMAPE_teste=18.5% | U_teste=8.955
  TARGET_EBITDA                       → RandomForest       SMAPE_CV=33.7% | SMAPE_teste=29.0% | U_teste=9.982
  TARGET_BPA_1                        → GradientBoosting   SMAPE_CV=9.2% | SMAPE_teste=4.5% | U_teste=2.573
  TARGET_BPA_1.01                     → GradientBoosting   SMAPE_CV=10.5% | SMAPE_teste=5.3% | U_teste=1.434
  TARGET_BPP_2.01                     → GradientBoosting   SMAPE_CV=14.2% | SMAPE_teste=7.1% | U_teste=3.657
  TARGET_BPP_2.03                     → GradientBoosting   SMAPE_CV=18.0% | SMAPE_teste=11.9% | U_teste=3.369
  TARGET_BPP_2                        → GradientBoosting   SMAPE_CV=9.2% | SMAPE_teste=4.5% | U_teste=2.573
  TARGET_DFC_MI_6.01                  → GradientBoostin

## Etapa 8. Feature importance e resíduos


In [11]:
def extrair_importancia(modelo, features):
    step = list(modelo.named_steps.keys())[-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)


print('\n=== Feature Importance — Melhor Modelo por Target ===')
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5 * n_t))
if n_t == 1:
    axes = [axes]

for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod = resultados[target][melhor_nome][0]
    feats_t = selected_features_por_target[target]
    imp = extrair_importancia(melhor_mod, feats_t)
    feature_importances[target] = {
        'algoritmo': melhor_nome,
        'features': feats_t,
        'importancias': imp.to_dict(),
    }

    ax = axes[i]
    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        ax.barh(range(len(top)), top.values[::-1], alpha=0.9)
        ax.set_yticks(range(len(top)))
        ax.set_yticklabels(top.index[::-1], fontsize=9)
        ax.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome} | SMAPE_teste={metricas_teste[target][melhor_nome]['SMAPE_macro_empresa']:.1%}",
                     fontsize=10, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            ax.text(v + imp.max() * 0.005, j, f'{v:.3f}', va='center', fontsize=8)
    else:
        ax.text(0.5, 0.5, 'Sem importância disponível', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()

plt.suptitle('Feature Importance — Melhor Modelo por Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Salvo: outputs/feature_importance.png')


print('\n=== Análise de Resíduos — Teste 2023–2024 ===')
cols_setor_disp = [c for c in teste.columns if c.startswith('setor_')]
fig, axes = plt.subplots(n_t, 2, figsize=(14, 5 * n_t))
if n_t == 1:
    axes = axes.reshape(1, -1)

for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod = resultados[target][melhor_nome][0]
    feats_t = selected_features_por_target[target]
    transformacao = get_target_transform(target)

    df_te = teste[feats_t + [target, 'CNPJ_CIA'] + (["DT_REFER"] if 'DT_REFER' in teste.columns else []) + cols_setor_disp].copy()
    df_te = df_te[df_te[target].notna()].copy()
    y_te = df_te[target].values
    y_pred = target_inverse_transform(melhor_mod.predict(df_te[feats_t].values), transformacao)
    residuos = y_te - y_pred

    ax1 = axes[i, 0]
    lim = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    ax1.scatter(y_pred, y_te, alpha=0.45, s=18, edgecolors='none')
    ax1.plot([-lim, lim], [-lim, lim], 'r--', lw=1.3)
    ax1.set_xlabel('Predito')
    ax1.set_ylabel('Observado')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\nPredito × Observado", fontsize=10, fontweight='bold')
    ax1.text(0.05, 0.92, f'R²m={metricas_teste[target][melhor_nome]["R2_macro_empresa"]:.3f}  SMAPE={metricas_teste[target][melhor_nome]["SMAPE_macro_empresa"]:.1%}',
             transform=ax1.transAxes, fontsize=8,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    ax2 = axes[i, 1]
    ax2.scatter(y_pred, residuos, alpha=0.45, s=18, edgecolors='none')
    ax2.axhline(0, color='r', lw=1.3, ls='--')
    ax2.axhline(np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.set_xlabel('Predito')
    ax2.set_ylabel('Resíduo')
    ax2.set_title(f'Resíduos × Predito | skew={pd.Series(residuos).skew():.2f}', fontsize=10, fontweight='bold')

plt.suptitle('Análise de Resíduos — Teste 2023–2024', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Salvo: outputs/analise_residuos.png')


=== Feature Importance — Melhor Modelo por Target ===
✅ Salvo: outputs/feature_importance.png

=== Análise de Resíduos — Teste 2023–2024 ===
✅ Salvo: outputs/analise_residuos.png


## Etapa 9. Persistência completa de artefatos

In [12]:
rows_cv, rows_te = [], []
for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({
            'Target': target,
            'Algoritmo': alg,
            'RMSE_CV_macro_empresa': m.get('RMSE_CV_macro_empresa'),
            'RMSE_CV_macro_empresa_std': m.get('RMSE_CV_macro_empresa_std'),
            'MAE_CV_macro_empresa': m.get('MAE_CV_macro_empresa'),
            'SMAPE_CV_macro_empresa': m.get('SMAPE_CV_macro_empresa'),
            'SMAPE_CV_macro_empresa_std': m.get('SMAPE_CV_macro_empresa_std'),
            'R2_CV_macro_empresa': m.get('R2_CV_macro_empresa'),
            'R2_CV_pooled': m.get('R2_CV_pooled'),
            'R2_within_CV': m.get('R2_within_CV'),
            'TheilU_CV_macro_empresa': m.get('TheilU_CV_macro_empresa'),
            'DA_CV_macro_empresa': m.get('DA_CV_macro_empresa'),
            'RMSE_CV_pooled': m.get('RMSE_CV_pooled'),
            'SMAPE_CV_pooled': m.get('SMAPE_CV_pooled'),
            'n_folds_wf': m.get('n_folds_wf'),
            'transformacao': m.get('transformacao'),
            'log_transform': m.get('log_transform'),
            'best_params': str(m.get('best_params')),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target': target,
            'Algoritmo': alg,
            'RMSE_teste_macro_empresa': mt.get('RMSE_macro_empresa'),
            'MAE_teste_macro_empresa': mt.get('MAE_macro_empresa'),
            'SMAPE_teste_macro_empresa': mt.get('SMAPE_macro_empresa'),
            'R2_teste_macro_empresa': mt.get('R2_macro_empresa'),
            'R2_teste_pooled': mt.get('R2_pooled'),
            'R2_teste_within': mt.get('R2_within'),
            'TheilU_teste_macro_empresa': mt.get('TheilU_macro_empresa'),
            'DA_teste_macro_empresa': mt.get('DA_macro_empresa'),
            'RMSE_teste_pooled': mt.get('RMSE_pooled'),
            'SMAPE_teste_pooled': mt.get('SMAPE_pooled'),
            'RMSE_baseline': b.get('RMSE_macro_empresa'),
            'Bateu_baseline': mt.get('RMSE_macro_empresa', np.inf) < b.get('RMSE_macro_empresa', np.inf),
            'TheilU_ok': (mt.get('TheilU_macro_empresa', 1.0) or 1.0) < 1.0,
        })


df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)

# predicoes detalhadas por linha
if predicoes_teste_detalhadas:
    df_pred = pd.concat(predicoes_teste_detalhadas, ignore_index=True)
else:
    df_pred = pd.DataFrame()

# salva csv/pkl/parquet
for name, obj in [
    ('resultados_cv.csv', df_cv),
    ('resultados_teste.csv', df_te),
]:
    obj.to_csv(PASTA_SAIDA / name, index=False)

if not df_pred.empty:
    df_pred.to_parquet(PASTA_SAIDA / 'predicoes_teste_detalhadas.parquet', index=False)
    df_pred.to_csv(PASTA_SAIDA / 'predicoes_teste_detalhadas.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl', 'wb') as f:
    pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl', 'wb') as f:
    pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl', 'wb') as f:
    pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f:
    pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl', 'wb') as f:
    pickle.dump(melhores, f)
with open(PASTA_SAIDA / 'selected_features_por_target.pkl', 'wb') as f:
    pickle.dump(selected_features_por_target, f)

relatorio = {
    'versao': 'V3_CompanyAware_WF_SMAPE',
    'data_execucao': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'ano_corte': ANO_CORTE,
    'n_treino': int(len(treino)),
    'n_teste': int(len(teste)),
    'targets': TARGETS,
    'algoritmos': list(ALGORITMOS.keys()),
    'n_features_originais': int(len(FEATURES)),
    'n_features_selecionadas_por_target': {t: len(v) for t, v in selected_features_por_target.items()},
    'train_dfp_only': TRAIN_DFP_ONLY,
    'corr_drop_threshold': CORR_DROP_THRESHOLD,
    'n_splits_wf': N_SPLITS_WF,
    'company_aware': True,
    'métricas_prioritárias': ['SMAPE_macro_empresa', 'TheilU_macro_empresa', 'DA_macro_empresa'],
    'selecao_modelo': 'menor SMAPE_CV_macro_empresa, desempate TheilU_CV_macro_empresa, desempate RMSE_CV_macro_empresa',
    'baseline': 'persistência do último valor da própria empresa',
    'feature_selection': 'treino-only + filtro de colinearidade',
    'pesos_amostrais': 'inverso por empresa e por target futuro repetido (DT_TARGET)',
    'results': {
        t: {
            alg: {
                'cv_smape_macro_empresa': float(resultados[t][alg][1].get('SMAPE_CV_macro_empresa', np.nan)) if resultados[t][alg][1].get('SMAPE_CV_macro_empresa') is not None else None,
                'test_smape_macro_empresa': float(metricas_teste[t][alg].get('SMAPE_macro_empresa', np.nan)) if metricas_teste[t][alg].get('SMAPE_macro_empresa') is not None else None,
                'test_theilu_macro_empresa': float(metricas_teste[t][alg].get('TheilU_macro_empresa', np.nan)) if metricas_teste[t][alg].get('TheilU_macro_empresa') is not None else None,
            }
            for alg in resultados[t].keys()
        }
        for t in resultados.keys()
    }
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print('\n' + '═' * 90)
print('RESUMO FINAL — Script 3 Company-Aware')
print('═' * 90)
print(f'Treino: {len(treino):,} obs | Teste: {len(teste):,} obs')
print(f'Features originais: {len(FEATURES)}')
print(f'Modelos treinados: {len(TARGETS) * len(ALGORITMOS)}')
print(f'CV: Walk-Forward {N_SPLITS_WF} folds')
print(f'Pesos amostrais: empresa + futuro repetido')
print(f'Flag COVID: {sorted(COVID_ANOS)}')
print('Artefatos salvos em outputs/')
print('  - modelo_<TARGET>_<ALG>.pkl')
print('  - resultados_cv.csv / resultados_teste.csv')
print('  - resultados_cv.pkl / metricas_teste.pkl / baselines.pkl')
print('  - feature_importances.pkl / melhores_modelos.pkl')
print('  - selected_features_por_target.pkl')
print('  - feature_importance.png / analise_residuos.png')
print('  - predicoes_teste_detalhadas.parquet / .csv')
print('  - relatorio_modelagem.json')
print('═' * 90)
print('✅ Pronto para o Script 4 (Avaliação + Z\'\' )')
print('═' * 90)

print('Artefatos salvos em outputs/')
print('  - modelos individuais por target/algoritmo')
print('  - resultados_cv.csv / resultados_teste.csv')
print('  - metricas_teste.pkl / baselines.pkl / melhores_modelos.pkl')
print('  - feature_importance.png / analise_residuos.png')
print('  - predicoes_teste_detalhadas.parquet / .csv')
print('  - relatorio_modelagem.json')
print('═' * 90)
print('✅ Pronto para o Script 4 (Avaliação + Z\'\' )')
print('═' * 90)



══════════════════════════════════════════════════════════════════════════════════════════
RESUMO FINAL — Script 3 Company-Aware
══════════════════════════════════════════════════════════════════════════════════════════
Treino: 713 obs | Teste: 168 obs
Features originais: 17
Modelos treinados: 36
CV: Walk-Forward 3 folds
Pesos amostrais: empresa + futuro repetido
Flag COVID: [2020, 2021]
Artefatos salvos em outputs/
  - modelo_<TARGET>_<ALG>.pkl
  - resultados_cv.csv / resultados_teste.csv
  - resultados_cv.pkl / metricas_teste.pkl / baselines.pkl
  - feature_importances.pkl / melhores_modelos.pkl
  - selected_features_por_target.pkl
  - feature_importance.png / analise_residuos.png
  - predicoes_teste_detalhadas.parquet / .csv
  - relatorio_modelagem.json
══════════════════════════════════════════════════════════════════════════════════════════
✅ Pronto para o Script 4 (Avaliação + Z'' )
══════════════════════════════════════════════════════════════════════════════════════════
Artefa